# Titanic Survival Prediction

**Author:** Vatsal Chandrani  
**Dataset:** Titanic passenger data (Kaggle)  
**Objective:** Build a logistic regression classifier to predict passenger survival based on features like age, class, gender, and fare

---

## Problem Statement

The sinking of the Titanic is one of the most infamous shipwrecks in history. This project uses machine learning to analyze what sorts of people were more likely to survive. We build a predictive model using passenger data including socio-economic status, age, gender, and other factors.


In [97]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

## Get the Data
Load the Titanic training dataset and inspect its structure.

In [ ]:
df = pd.read_csv('../data/titanic_train.csv')

**Check the first few rows of the dataset**

In [ ]:
df.head()

## Exploratory Data Analysis

Understand the dataset's structure, distributions, and relationships between features and the survival target.

In [ ]:
df.info()

In [ ]:
df.describe()

### Missing Data

Identify which columns have missing values before cleaning.

In [ ]:
sns.set_style('whitegrid')
plt.figure(figsize=(10, 6))
sns.heatmap(df.isnull(), yticklabels=False, cbar=False, cmap='viridis')
plt.title('Missing Data Heatmap')

### Survival by Gender

Gender was the strongest predictor of survival due to the "women and children first" evacuation protocol.

In [ ]:
sns.countplot(x='Survived', hue='Sex', data=df, palette='RdBu_r')
plt.title('Survival Count by Gender')

### Survival by Passenger Class

First-class passengers had access to upper decks and lifeboats, giving them significantly better survival odds.

In [ ]:
sns.countplot(x='Survived', hue='Pclass', data=df, palette='rainbow')
plt.title('Survival Count by Passenger Class')

### Age Distribution

In [ ]:
df['Age'].hist(bins=30, color='steelblue', edgecolor='black', alpha=0.7)
plt.xlabel('Age')
plt.title('Age Distribution of Passengers')

### Fare Distribution by Passenger Class

In [ ]:
df.boxplot(column='Fare', by='Pclass', figsize=(8, 5))
plt.title('Fare Distribution by Passenger Class')
plt.suptitle('')

## Feature Engineering

Handle missing values and encode categorical variables to prepare data for the model.

### Impute Missing Age

Fill missing Age values using the median age within each passenger class — more accurate than the overall median.

In [ ]:
def fill_age(cols):
    age = cols[0]
    pclass = cols[1]
    if pd.isnull(age):
        return df.groupby('Pclass')['Age'].median()[pclass]
    return age

df['Age'] = df[['Age', 'Pclass']].apply(fill_age, axis=1)

In [ ]:
# Drop Cabin (77% missing) and rows with missing Embarked
df.drop('Cabin', axis=1, inplace=True)
df.dropna(inplace=True)

In [ ]:
sex = pd.get_dummies(df['Sex'], drop_first=True)
embark = pd.get_dummies(df['Embarked'], drop_first=True)

df = pd.concat([df, sex, embark], axis=1)
df.drop(['Sex', 'Embarked', 'Name', 'Ticket', 'PassengerId'], axis=1, inplace=True)
df.head()

## Model Building

Split the data and train a logistic regression classifier.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('Survived', axis=1)
y = df['Survived']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

## Training the Model

In [ ]:
from sklearn.linear_model import LogisticRegression

logmodel = LogisticRegression(max_iter=200)
logmodel.fit(X_train, y_train)

## Predictions and Evaluation

In [ ]:
predictions = logmodel.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, predictions))

In [ ]:
cm = confusion_matrix(y_test, predictions)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did not survive', 'Survived'],
            yticklabels=['Did not survive', 'Survived'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix')

---

## Key Findings

### Model Performance
- Logistic regression classifier achieved strong predictive accuracy on the held-out test set
- Model correctly identifies both survivors and non-survivors with balanced precision and recall

### Survival Insights
1. **Gender**: Female passengers had significantly higher survival rates — strongest single predictor
2. **Passenger Class**: First-class passengers survived at much higher rates than third-class
3. **Age**: Children showed higher survival rates, especially in upper classes
4. **Fare**: Higher fares (correlated with class) positively associated with survival

**Historical Context**: These findings align with historical accounts of "women and children first" evacuation protocols and class-based lifeboat access.

---

**Technologies Used**: Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn  
**Skills Demonstrated**: Logistic regression, binary classification, missing data imputation, feature engineering, model evaluation